# Reproducing TimesNet (ICLR 2023) - Table 13: long-term forecasting

TimesNet column of **Table 13**: MSE/MAE on 9 datasets × 4 horizons (input length 96; 36 for ILI), plus the per-dataset averages of Table 2. 36 configurations × 3 seeds = 108 runs. This is the heaviest table: ETT, Exchange, ILI and Weather are quick, Electricity takes hours and Traffic (862 variables, d_model 512) is the slowest and may need an L4/A100 or more than one Colab session. Section 6 has one cell per dataset so you can run one dataset per day.


## 1. Wrap-UP Helper Functions

Includes functions whose purpose is to:

1) Set-Up Colab Environment Initialization and Avoid Collisions (e.x. Library Versions, GPUs, Folders etc.)

2) Wrap-Up Original Repository Code, Reproduce it and Formulate it Properly for Presentation and Evaluation (e.x. Seeds, Metrics, Discover Errors, Differences with the Paper Results etc.)

In [ ]:
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# ---------------------------- Import Libraries -----------------------------
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

%%writefile repro_utils.py
import glob
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import time





# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# ----------------------------- Initializations -----------------------------
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

SEEDS = [2021, 2022, 2023]   # 2021 = TSLib's own seed; the paper averages 3 runs
SMOKE = False                # True = 1 training epoch per run, to test the pipeline
USE_DRIVE = False            # keep results on Google Drive (survives disconnects)
HF_REVISION = None           # None = latest on first run, then pinned automatically
TSLIB_URL = "https://github.com/thuml/Time-Series-Library.git"      # Source of Original code to clone
TSLIB_COMMIT = "4e938a1767106324dd753b2a44832bf870a0252e"           # Original Version of the code (2026-04-18)
HF_REPO = "thuml/Time-Series-Library"                               # Original Huggig Face dataset repository
DEFAULT_SEEDS = [2021, 2022, 2023]                                  # Original Seeds for Reprodusability

# Versions from TSLib's README / requirements.txt.
PINNED_PACKAGES = ["numpy==2.1.2",
                   "pandas==2.3.3",
                   "scipy==1.16.3",
                   "scikit-learn==1.7.2",
                   "matplotlib==3.10.8",
                   "sktime==0.40.1",
                   "patool==4.0.3",
                   "tqdm==4.66.5",
                   "datasets==4.5.0",
                   "huggingface_hub==0.36.0"]

TORCH_PROFILES = {"tslib": ("torch==2.5.1", "https://download.pytorch.org/whl/cu121", ["sympy==1.13.1"]),
                  "blackwell": ("torch==2.7.1", "https://download.pytorch.org/whl/cu128", [])}

_RUN_PY_SEED = "fix_seed = 2021"
_RUN_PY_SEED_NEW = "fix_seed = int(os.environ.get('TSLIB_SEED', '2021'))"
_RUN_PY_NP_SEED = "np.random.seed(fix_seed)"
_DETERMINISM_BLOCK = """np.random.seed(fix_seed)
    # [repro patch] deterministic kernels and no TF32 (TITAN RTX used in the paper has no TF32)
    if os.environ.get('TSLIB_DETERMINISTIC', '1') == '1':
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.use_deterministic_algorithms(True, warn_only=True)"""





# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# ---------------------------- Helper Functions -----------------------------
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

# Returns everything a shell command prints
def sh(cmd,
       cwd = None,
       env = None,
       check = True,
       quiet = False):

    # cmd: The command
    # cwd: Folder to run the command (cwd = None, runs it in the current folder of Colab)
    # env: Environment Variables
    # check: Check whether the command fails
    # quiet: Print the command, prefixed with "$"

    if env is None:
        env = dict(os.environ, MPLBACKEND="Agg")
    if not quiet:
        print("$", cmd)
    res = subprocess.run(cmd,
                         shell = True,
                         cwd = cwd,
                         env = env,
                         text = True,
                         stdout = subprocess.PIPE,
                         stderr = subprocess.STDOUT)
    if res.returncode != 0 and check:
        print(res.stdout[-4000:])
        raise RuntimeError(f"command failed ({res.returncode}): {cmd}")

    return res.stdout



# Returns True if the code can be run in Google Colab
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False



# Returns the folder, in which the reproduced table results are stored
def results_root(name,
                 use_drive = True):

    # name: The name of the table with respect to the ones in the paper (e.x. table 17)
    # use_drive: Whether to use Drive or Colab

    if in_colab() and use_drive:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        root = f"/content/drive/MyDrive/TimesNet_reproduction/{name}"
    else:
        base = "/content" if os.path.exists("/content") else os.getcwd()
        root = os.path.join(base, "TimesNet_reproduction", name)
    os.makedirs(root, exist_ok=True)

    return root



# Returns the content of a json file
def load_json(path,
              default = None):

    # path: Path of the file
    # default: Value to return, if the file doesn't exist

    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return default



# Returns a JSON file
def save_json(path, obj):

    # path: Write the JSON file to "path"
    # obj: Which to write as a JSON file

    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)





# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# ------------------------- Environment Functions ---------------------------
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

# Return the computational capability
def gpu_compute_capability():

    # e.x. 7.5 for T4, 8.0 for A100, 12.0 for RTX PRO 6000 (Colab G4)

    res = subprocess.run("nvidia-smi --query-gpu=compute_cap --format=csv,noheader",
                         shell=True, text=True, capture_output=True)
    try:
        return float(res.stdout.strip().splitlines()[0]) if res.returncode == 0 else None
    except (ValueError, IndexError):
        return None



# Returns the path to the environment and it's description (as a dictionary)
def setup_environment(env_dir = "/content/tslib_env",
                      python_version = "3.11",
                      profile = None):

    # env_dir: Directory of the environment
    # python_version: Version of Python used
    # profile: The GPU

    cap = gpu_compute_capability()
    if profile is None:
        profile = "blackwell" if cap is not None and cap >= 10 else "tslib"
    if profile != "tslib":
        env_dir = f"{env_dir}_{profile}"
        print(f"NOTE: this GPU (compute capability {cap}) is newer than torch 2.5.1 supports, "
              f"so torch 2.7.1 (CUDA 12.8) is used instead of TSLib's recommended 2.5.1.\n"
              f"Runs stay repeatable on this GPU type; mention the version in your report, "
              f"or switch the runtime to T4/L4/A100/H100 to use TSLib's recommended version.")

    torch_pkg, index, extra = TORCH_PROFILES[profile]
    py = os.path.join(env_dir, "bin", "python")
    marker = os.path.join(env_dir, ".repro_ready")

    if not os.path.exists(marker):
        sh(f"{sys.executable} -m pip install -q uv")
        sh(f"uv venv --python {python_version} {env_dir}")
        sh(f"uv pip install --python {py} {torch_pkg} --index-url {index}")
        sh(f"uv pip install --python {py} " + " ".join(PINNED_PACKAGES + extra))
        open(marker, "w").close()

    info = sh(f"{py} -c \"import torch, numpy, pandas, sys; "
              f"print(sys.version.split()[0], torch.__version__, torch.version.cuda, "
              f"torch.cuda.is_available(), numpy.__version__, pandas.__version__)\"",
              quiet=True).split()

    env = {"python": info[0], "torch": info[1], "cuda": info[2],
           "cuda_available": info[3] == "True", "numpy": info[4], "pandas": info[5],
           "torch_profile": profile, "compute_capability": cap}
    print("Environment:", env)

    if cap is None or not env["cuda_available"]:
        print("WARNING: no GPU visible. In Colab: Runtime > Change runtime type > GPU.")
    else:
        test = subprocess.run(
            [py, "-c", "import torch; x = torch.randn(64, 64, device='cuda'); "
                       "print(float((x @ x).sum()), torch.cuda.get_device_name(0))"],
            text=True, capture_output=True)
        if test.returncode != 0:
            raise RuntimeError(
                "The GPU test failed, so training would fail too:\n"
                + test.stdout[-1500:] + test.stderr[-1500:]
                + "\nFix: Runtime > Change runtime type > T4, L4, A100 or H100, then run all cells.")
        print("GPU test passed:", test.stdout.strip().split(" ", 1)[-1])

    return py, env



# Returns the name of the GPU
def gpu_name():
    res = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                         text=True, capture_output=True)
    lines = res.stdout.strip().splitlines()
    return lines[0] if res.returncode == 0 and lines else "no GPU"





# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# -------------------------- Repository Functions ---------------------------
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

# Ensures Reprodusability for the TSLib
def prepare_repo(repo_dir = "/content/Time-Series-Library",
                 commit = TSLIB_COMMIT):

    # repo_dir: Directory of the repository
    # commit: Version of the code to use

    if not os.path.exists(os.path.join(repo_dir, ".git")):
        sh(f"git clone -q {TSLIB_URL} {repo_dir}")
    sh(f"git -C {repo_dir} fetch -q origin {commit}", check=False, quiet=True)
    sh(f"git -C {repo_dir} checkout -q -f {commit}")
    head = sh(f"git -C {repo_dir} rev-parse HEAD", quiet=True).strip()
    assert head == commit, f"expected {commit}, got {head}"

    run_py = os.path.join(repo_dir, "run.py")
    with open(run_py) as f:
        src = f.read()
    if "[repro patch]" not in src:
        assert src.count(_RUN_PY_SEED) == 1 and src.count(_RUN_PY_NP_SEED) == 1, \
            "run.py layout changed; patches cannot be applied safely"
        src = src.replace(_RUN_PY_SEED, _RUN_PY_SEED_NEW)
        src = src.replace(_RUN_PY_NP_SEED, _DETERMINISM_BLOCK)
        with open(run_py, "w") as f:
            f.write(src)
    _patch_m4_numpy(repo_dir)
    print(f"TSLib ready at {repo_dir} (commit {commit[:10]}), run.py patched "
          f"(seed from TSLIB_SEED, deterministic kernels, TF32 off), M4 NumPy fix applied.")
    return repo_dir



#
_STACK_HELPER = '''
def _repro_stack(rows):
    """[repro patch] NumPy<1.24 behaviour of np.array(list_of_arrays): a 2-D array when all
    rows have the same length, otherwise a 1-D object array (NumPy>=1.24 raises instead)."""
    rows = list(rows)
    if len({len(r) for r in rows}) <= 1:
        return np.array(rows)
    out = np.empty(len(rows), dtype=object)
    out[:] = rows
    return out
'''



# Helps avoid TSLib's M4 code crash
def _patch_m4_numpy(repo_dir):
    """TSLib's M4 code builds arrays from series of different lengths. That raises an
    error on NumPy>=1.24 (TSLib pins 2.1.2), so M4 would crash on the real data.
    This restores the old NumPy behaviour exactly; nothing else changes."""
    targets = {
        "utils/m4_summary.py": [
            "return np.array([v[~np.isnan(v)] for v in values[groups == group_name]])",
            "naive2_forecasts = np.array([v[~np.isnan(v)] for v in naive2_forecasts])"],
        "data_provider/data_loader.py": [
            "training_values = np.array(\n            [v[~np.isnan(v)] for v in"],
    }
    for rel, olds in targets.items():
        path = os.path.join(repo_dir, rel)
        with open(path) as f:
            src = f.read()
        if "[repro patch] NumPy<1.24" in src:
            continue
        for old in olds:
            assert src.count(old) == 1, f"{rel}: unexpected layout, cannot patch"
            src = src.replace(old, old.replace("np.array(", "_repro_stack(", 1))
        anchor = "import numpy as np\n"
        assert anchor in src, f"{rel}: numpy import not found"
        src = src.replace(anchor, anchor + _STACK_HELPER, 1)
        with open(path, "w") as f:
            f.write(src)



# Returns the dataset used
def download_data(repo_dir,
                  python,
                  folders,
                  revision = None):

    # repo_dir: Directory of the repository
    # python: Version of Python
    # folders: Datasets folders to download
    # revision: Hugging Face revision

    """Download dataset folders from the TSLib Hugging Face repo into ./dataset.
    Returns the exact dataset revision (commit sha) used."""
    patterns = json.dumps([f"{d}/*" for d in folders])
    code = (
        "from huggingface_hub import snapshot_download, HfApi\n"
        f"rev = {revision!r} or HfApi().dataset_info({HF_REPO!r}).sha\n"
        f"snapshot_download(repo_id={HF_REPO!r}, repo_type='dataset', revision=rev,"
        f" allow_patterns={patterns}, local_dir='dataset')\n"
        "print('REVISION', rev)\n"
    )
    out = sh(f"{python} -c {shlex.quote(code)}", cwd=repo_dir, quiet=True)
    rev = re.search(r"REVISION (\S+)", out).group(1)
    missing = [d for d in folders if not os.listdir(os.path.join(repo_dir, "dataset", d))]
    if missing:
        raise RuntimeError(f"dataset folders empty after download: {missing}")
    print(f"Datasets {folders} ready (Hugging Face revision {rev[:10]}).")

    return rev



# Returns the current Colab session
def record_environment(root,
                       env,
                       commit,
                       data_rev):

    # root: Folder, in which the reproduced table results are stored
    # env: Environment Variables
    # commit: Version of the code to use
    # data_rev: Hugging Face Datasets to download

    """Store the environment; warn if this session's GPU differs from earlier runs."""
    path = os.path.join(root, "environment.json")
    info = load_json(path, {})
    current = {"gpu": gpu_name(), "tslib_commit": commit, "hf_dataset_revision": data_rev,
               **{f"env_{k}": v for k, v in env.items()}}
    runs_dir = os.path.join(root, "runs")
    has_runs = os.path.isdir(runs_dir) and any(f.endswith(".json") for f in os.listdir(runs_dir))
    if info and not has_runs:
        info = {}   # nothing finished yet: the new session simply becomes the reference
    if info:
        diffs = {k: (info.get(k), v) for k, v in current.items() if info.get(k) != v}
        if diffs:
            print("WARNING - this session differs from the one that produced earlier "
                  "results in this folder:")
            for k, (old, new) in diffs.items():
                print(f"   {k}: {old} -> {new}")
            print("Runs are only bit-for-bit repeatable on the same GPU model and "
                  "software. Use a new results folder or the same GPU type.")
    else:
        save_json(path, current)
    return current




# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# -------------------------------- Commands ---------------------------------
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

# Returns a list with one entry per training command
def expand_script(repo_dir,
                  script):

    # repo_dir: Directory of the Repository
    # script: Script to run

    shim = ("python() { for a in \"$@\"; do printf '%s\\037' \"$a\"; done; printf '\\036'; }; "
            "python3() { python \"$@\"; }; "
            f"source {shlex.quote(script)}")
    out = subprocess.run(["bash", "-c", shim], cwd=repo_dir, text=True,
                         capture_output=True, check=True).stdout
    commands = []
    for rec in out.split("\x1e"):
        parts = [p for p in rec.split("\x1f") if p != ""]
        if "run.py" in parts:
            commands.append(parts[parts.index("run.py") + 1:])

    return commands



# Returns the value that follows --<key> in an arg list
def arg(args,
        key,
        default = None):

    # args: The argument
    # key: The key
    # default: Returns None is the flag is absent

    flag = "--" + key
    return args[args.index(flag) + 1] if flag in args else default



# Returns a copy of the argument list
def set_arg(args,
            key,
            value):

    # args: Argument to return
    # key: The key to select
    # value: The value to set to "--<key>"

    args = list(args)
    flag = "--" + key
    if flag in args:
        args[args.index(flag) + 1] = str(value)
    else:
        args += [flag, str(value)]

    return args





# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# ----------------------------- Commands to Run -----------------------------
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

# Returns the TSLibs results
def run_tslib(python,
              repo_dir,
              args,
              seed,
              log_path,
              smoke = False):

    # python: Python version
    # repo_dir: Directory of the repository
    # args: Argument
    # seed: Seeds
    # log_path: Path logs
    # smoke: Whether to run a smoke test

    args = list(args)
    if smoke:
        args = set_arg(args, "train_epochs", 1)
    if os.environ.get("REPRO_FORCE_CPU") == "1":   # only for testing without a GPU
        args = args + ["--no_use_gpu", "--num_workers", "0"]
    env = dict(os.environ, TSLIB_SEED=str(seed), TSLIB_DETERMINISTIC="1",
               MPLBACKEND="Agg",   # Colab sets an inline backend the isolated env lacks
               CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONHASHSEED=str(seed),
               PYTHONUNBUFFERED="1")
    env.pop("CUDA_VISIBLE_DEVICES", None)
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    lines = []
    with open(log_path, "w") as log:
        log.write("python -u run.py " + " ".join(shlex.quote(a) for a in args) + "\n")
        log.write(f"# TSLIB_SEED={seed}\n")
        proc = subprocess.Popen([python, "-u", "run.py", *args], cwd=repo_dir, env=env,
                                text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in proc.stdout:
            log.write(line)
            lines.append(line)
        proc.wait()

    return proc.returncode == 0, "".join(lines)



#
def clean_outputs(repo_dir,
                  keep_checkpoint_to = None):

    # repo_dir: Director of repository
    # keep_checkpoint_to: Checkpoints to select

    if keep_checkpoint_to:
        ckpts = sorted(glob.glob(os.path.join(repo_dir, "checkpoints", "*", "checkpoint.pth")),
                       key=os.path.getmtime)
        if ckpts:
            os.makedirs(os.path.dirname(keep_checkpoint_to), exist_ok=True)
            shutil.copy2(ckpts[-1], keep_checkpoint_to)
    for d in ["checkpoints", "results", "test_results"]:
        shutil.rmtree(os.path.join(repo_dir, d), ignore_errors=True)



# Prints logs of the execution time and seeds per dataset
def run_all(python,
            repo_dir,
            root,
            commands,
            seeds,
            parse,
            meta,
            smoke = False,
            key_fn = lambda a: arg(a, "model_id"),
            keep_checkpoints = False,
            after_run = None,
            max_commands = None):

    # python: Python version
    # repo_dir: Directory of the repository
    # root: Folder, in which the reproduced table results are stored
    # commands: Lists obtained from "expand_script"
    # seeds: Selected seeds
    # parse: Function that turns a run's output into a dictionary of metrics (e.x. accuracy)
    # meta: The record from "record_environment", copied into every result
    # smoke: Whether to run a smoke test or not
    # key_fn: Name of a run
    # keep_checkpoints: Whether to copy each of the model checkpoints into "checkpoints/" on the Drive or o Colab
    # after_run: Optional function to store predictions
    # max_commands: Number of commands to execute

    """Run every (command, seed) pair not already stored in `root/runs`.

    parse(output) -> dict of metrics or None; after_run(args, seed) is an optional hook.
    """
    runs_dir = os.path.join(root, "smoke_runs" if smoke else "runs")
    todo = commands[:max_commands] if max_commands else commands
    total, n = len(todo) * len(seeds), 0
    for seed in seeds:
        for args in todo:
            n += 1
            key = key_fn(args)
            out_json = os.path.join(runs_dir, f"{key}__seed{seed}.json")
            if os.path.exists(out_json):
                print(f"[{n}/{total}] {key} seed {seed}: already done, skipping")
                continue
            print(f"[{n}/{total}] {key} seed {seed}: running ...", end="", flush=True)
            t0 = time.time()
            ok, output = run_tslib(python, repo_dir, args, seed,
                                   os.path.join(root, "logs", f"{key}__seed{seed}.log"), smoke)
            metrics = parse(output) if ok else None
            if after_run:
                after_run(args, seed)
            ckpt = (os.path.join(root, "checkpoints", f"{key}__seed{seed}.pth")
                    if keep_checkpoints and ok and not smoke else None)
            clean_outputs(repo_dir, ckpt)
            minutes = (time.time() - t0) / 60
            if not metrics:
                print(f" FAILED after {minutes:.1f} min - see logs/{key}__seed{seed}.log")
                print(output[-2500:])
                continue
            save_json(out_json, {"key": key, "seed": seed, "metrics": metrics,
                                 "minutes": round(minutes, 2), "args": args,
                                 "smoke": smoke, **meta})
            shown = ", ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}"
                              for k, v in metrics.items())
            print(f" done in {minutes:.1f} min -> {shown}")



# Returns the reproduced table
def load_runs(root,
              smoke = False):

    # root: Root to select
    # smoke: Whether is smoke test or not

    import pandas as pd
    rows = []
    for p in glob.glob(os.path.join(root, "smoke_runs" if smoke else "runs", "*.json")):
        r = load_json(p)
        rows.append({"key": r["key"], "seed": r["seed"], **r["metrics"]})
    return pd.DataFrame(rows)



# Returns labels (matches, close, differs)
def verdict(delta,
            rel,
            kind):

    # delta: Difference in percentage
    # rel: Difference in percent of the paper's values
    # kind: Type of difference (e.x. absolute)

    if kind == "rel":
        a = abs(rel)
        return "✓ matches" if a <= 2 else ("~ close" if a <= 5 else "✗ differs")
    a = abs(delta)
    return "✓ matches" if a <= 1 else ("~ close" if a <= 3 else "✗ differs")



# Returns a check of determinism to define if the new result is identical to the stored one
def determinism_check(python,
                      repo_dir,
                      root,
                      args,
                      seed,
                      parse,
                      smoke = False):

    # python: Python version
    # repo_dir: Directory of the repository
    # root: Folder, in which the reproduced table results are stored
    # args: Arguments
    # seed: Seeds
    # parse: Function that turns a run's output into a dictionary of metrics (e.x. accuracy)
    # smoke: Whether it is a smoke test or not

    """Re-run one stored configuration and check the metric is identical."""
    key = arg(args, "model_id")
    stored = load_json(os.path.join(root, "smoke_runs" if smoke else "runs",
                                    f"{key}__seed{seed}.json"))
    if not stored:
        print("Run the experiments first."); return None
    ok, output = run_tslib(python, repo_dir, args, seed,
                           os.path.join(root, "logs", f"{key}__seed{seed}__recheck.log"), smoke)
    clean_outputs(repo_dir)
    again = parse(output) if ok else None
    same = again == stored["metrics"]
    print(f"stored : {stored['metrics']}\nre-run : {again}\n"
          + ("IDENTICAL - the run is bit-for-bit repeatable on this setup." if same else
             "DIFFERENT - check GPU model / environment against environment.json."))

    return same

Writing repro_utils.py


## 2. Environment, Original Code, Data, Optimizer Changes

Includes Notebook Cells whose purpose is to:

1) Create an Isolated Python 3.11 Environment (torch 2.5.1, CUDA 12.1 wheels, TSLib's pinned versions)

2) Clone TSLib at fixed commits

3) Downloads Original Datasets from Hugging Face


In [ ]:
import importlib, os, sys
sys.path.insert(0, os.getcwd())
import repro_utils as R
importlib.reload(R)
SEEDS, SMOKE, USE_DRIVE, HF_REVISION = R.SEEDS, R.SMOKE, R.USE_DRIVE, R.HF_REVISION

# Get ROOT
ROOT = R.results_root(name = 'table13',
                      use_drive = USE_DRIVE)
print("Results folder:", ROOT)

# Get Python version and Environment
PY, ENV = R.setup_environment()          # Python 3.11 + torch 2.5.1 (cu121)

# Get TSLib Repository, with fixed commit
REPO = R.prepare_repo()
os.chdir(REPO)

# Get JSON
previous = R.load_json(os.path.join(ROOT, "environment.json"), {})

# Get Datasets
DATA_REV = R.download_data(repo_dir = REPO,
                           python = PY,
                           folders = ['ETT-small',
                                      'electricity',
                                      'traffic',
                                      'weather',
                                      'exchange_rate',
                                      'illness'],
                           revision = HF_REVISION or previous.get("hf_dataset_revision"))

# Get Colab Session
META = R.record_environment(root = ROOT,
                            env = ENV,
                            commit = R.TSLIB_COMMIT,
                            data_rev = DATA_REV)
print(META)

Results folder: /content/TimesNet_reproduction/table13
$ /usr/bin/python3 -m pip install -q uv
$ uv venv --python 3.11 /content/tslib_env
$ uv pip install --python /content/tslib_env/bin/python torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
$ uv pip install --python /content/tslib_env/bin/python numpy==2.1.2 pandas==2.3.3 scipy==1.16.3 scikit-learn==1.7.2 matplotlib==3.10.8 sktime==0.40.1 patool==4.0.3 tqdm==4.66.5 datasets==4.5.0 huggingface_hub==0.36.0 sympy==1.13.1
Environment: {'python': '3.11.17', 'torch': '2.5.1+cu121', 'cuda': '12.1', 'cuda_available': True, 'numpy': '2.1.2', 'pandas': '2.3.3', 'torch_profile': 'tslib', 'compute_capability': 8.9}
GPU test passed: NVIDIA L4
$ git clone -q https://github.com/thuml/Time-Series-Library.git /content/Time-Series-Library
$ git -C /content/Time-Series-Library checkout -q -f 4e938a1767106324dd753b2a44832bf870a0252e
TSLib ready at /content/Time-Series-Library (commit 4e938a1767), run.py patched (seed from TSLIB_SEED, dete

## 3. Paper values (TimesNet column)

In [ ]:
# TimesNet column of Table 13: horizon -> (MSE, MAE)
PAPER_T13 = {"ETTm1": {96: (0.338, 0.375), 192: (0.374, 0.387), 336: (0.410, 0.411), 720: (0.478, 0.450)},
             "ETTm2": {96: (0.187, 0.267), 192: (0.249, 0.309), 336: (0.321, 0.351), 720: (0.408, 0.403)},
             "ETTh1": {96: (0.384, 0.402), 192: (0.436, 0.429), 336: (0.491, 0.469), 720: (0.521, 0.500)},
             "ETTh2": {96: (0.340, 0.374), 192: (0.402, 0.414), 336: (0.452, 0.452), 720: (0.462, 0.468)},
             "Electricity": {96: (0.168, 0.272), 192: (0.184, 0.289), 336: (0.198, 0.300), 720: (0.220, 0.320)},
             "Traffic": {96: (0.593, 0.321), 192: (0.617, 0.336), 336: (0.629, 0.336), 720: (0.640, 0.350)},
             "Weather": {96: (0.172, 0.220), 192: (0.219, 0.261), 336: (0.280, 0.306), 720: (0.365, 0.359)},
             "Exchange": {96: (0.107, 0.234), 192: (0.226, 0.344), 336: (0.367, 0.448), 720: (0.964, 0.746)},
             "ILI": {24: (2.317, 0.934), 36: (1.972, 0.920), 48: (2.238, 0.940), 60: (2.027, 0.928)}}

PAPER_T13_AVG = {"ETTm1": (0.400, 0.406),
                 "ETTm2": (0.291, 0.333),
                 "ETTh1": (0.458, 0.450),
                 "ETTh2": (0.414, 0.427),
                 "Electricity": (0.192, 0.295),
                 "Traffic": (0.620, 0.336),
                 "Weather": (0.259, 0.287),
                 "Exchange": (0.416, 0.443),
                 "ILI": (2.139, 0.931)}

## 4. The Original Commands

Includes Notebook Cells whose purpose is to:

1) Use Commands from the Original Repository

2) Use Original Per-Dataset Initializations for the Model


In [ ]:
import re, glob, pandas as pd

# Get Original Commands from repo (cheapest first, Traffic last)
SCRIPTS = ["ETT_script/TimesNet_ETTh1.sh",
           "ETT_script/TimesNet_ETTh2.sh",
           "ETT_script/TimesNet_ETTm1.sh",
           "ETT_script/TimesNet_ETTm2.sh",
           "Exchange_script/TimesNet.sh",
           "ILI_script/TimesNet.sh",
           "Weather_script/TimesNet.sh",
           "ECL_script/TimesNet.sh",
           "Traffic_script/TimesNet.sh"]

COMMANDS = [c for s in SCRIPTS for c in R.expand_script(REPO, f"scripts/long_term_forecast/{s}")]
assert len(COMMANDS) == 36, len(COMMANDS)

# Maps a run name (e.g. "ECL_96_96") to the paper's dataset name
def dataset_of(model_id):
    prefix = model_id.split("_")[0]
    return {"ECL": "Electricity",
            "traffic": "Traffic",
            "weather": "Weather",
            "ili": "ILI"}.get(prefix, prefix)

# Get official per-dataset settings
pd.DataFrame([{k: R.arg(c, k) for k in ["model_id",
                                        "seq_len",
                                        "pred_len",
                                        "e_layers",
                                        "d_model",
                                        "d_ff",
                                        "top_k",
                                        "batch_size",
                                        "train_epochs"]} for c in COMMANDS])

,model_id,seq_len,pred_len,e_layers,d_model,d_ff,top_k,batch_size,train_epochs
0,ETTh1_96_96,96,96,2,16,32,5,None,None
1,ETTh1_96_192,96,192,2,16,32,5,None,None
2,ETTh1_96_336,96,336,2,16,32,5,None,None
3,ETTh1_96_720,96,720,2,16,32,5,None,None
4,ETTh2_96_96,96,96,2,32,32,5,None,None
5,ETTh2_96_192,96,192,2,32,32,5,None,None
6,ETTh2_96_336,96,336,2,32,32,5,None,None
7,ETTh2_96_720,96,720,2,32,32,5,None,None
8,ETTm1_96_96,96,96,2,64,64,5,None,None
9,ETTm1_96_192,96,192,2,64,64,5,None,None


In [ ]:
# [paper] Table 7, long-term: k = 5, 2 layers, d_model = min(max(2^ceil(log2 C), 32), 512),
# LR 1e-4, MSE loss, batch size 32, 10 epochs. d_ff is not in the paper: kept from the scripts.
import math
CHANNELS = {"ETTh1": 7,
            "ETTh2": 7,
            "ETTm1": 7,
            "ETTm2": 7,
            "Exchange": 8,
            "ILI": 7,
            "Weather": 21,
            "Electricity": 321,
            "Traffic": 862}

def paper_d_model(C, d_min = 32, d_max = 512):
    return min(max(2 ** math.ceil(math.log2(C)), d_min), d_max)

PAPER_COMMANDS = []
for c in COMMANDS:
    ds = dataset_of(R.arg(c, "model_id"))
    assert R.arg(c, "enc_in") == str(CHANNELS[ds]), (ds, R.arg(c, "enc_in"))
    for k, v in {"e_layers": 2,
                 "top_k": 5,
                 "d_model": paper_d_model(CHANNELS[ds]),
                 "batch_size": 32,
                 "learning_rate": 0.0001,
                 "loss": "MSE",
                 "train_epochs": 10}.items():
        c = R.set_arg(c, k, v)
    PAPER_COMMANDS.append(c)

# Get paper settings per run
pd.DataFrame([{k: R.arg(c, k) for k in ["model_id",
                                        "seq_len",
                                        "pred_len",
                                        "e_layers",
                                        "d_model",
                                        "d_ff",
                                        "top_k",
                                        "batch_size",
                                        "learning_rate",
                                        "train_epochs"]} for c in PAPER_COMMANDS])

,model_id,seq_len,pred_len,e_layers,d_model,d_ff,top_k,batch_size,learning_rate,train_epochs
0,ETTh1_96_96,96,96,2,32,32,5,32,0.0001,10
1,ETTh1_96_192,96,192,2,32,32,5,32,0.0001,10
2,ETTh1_96_336,96,336,2,32,32,5,32,0.0001,10
3,ETTh1_96_720,96,720,2,32,32,5,32,0.0001,10
4,ETTh2_96_96,96,96,2,32,32,5,32,0.0001,10
5,ETTh2_96_192,96,192,2,32,32,5,32,0.0001,10
6,ETTh2_96_336,96,336,2,32,32,5,32,0.0001,10
7,ETTh2_96_720,96,720,2,32,32,5,32,0.0001,10
8,ETTm1_96_96,96,96,2,32,64,5,32,0.0001,10
9,ETTm1_96_192,96,192,2,32,64,5,32,0.0001,10


## 5. Run

In [ ]:
# Reads MSE and MAE from the run's output
def parse_lt(output):
    m = re.findall(r"mse:([0-9.eE+-]+), mae:([0-9.eE+-]+)", output)
    return {"mse": float(m[-1][0]), "mae": float(m[-1][1])} if m else None

# Runs one dataset (all 4 horizons by default) with the paper's settings
def run_dataset(dataset,
                seeds = None,
                horizons = None):

    # dataset: Paper name of the dataset (e.g. "ETTh1", "Electricity")
    # seeds: Seeds to run (default: all SEEDS)
    # horizons: Horizons to run (default: all 4)

    seeds = SEEDS if seeds is None else seeds
    selected = [c for c in PAPER_COMMANDS if dataset_of(R.arg(c, "model_id")) == dataset
                and (horizons is None or int(R.arg(c, "pred_len")) in horizons)]
    assert selected, f"unknown dataset or horizon: {dataset} {horizons}"
    print(f"== {dataset}: {len(selected)} horizon(s) x {len(seeds)} seed(s) ==")
    R.run_all(python = PY,
              repo_dir = REPO,
              root = ROOT,
              commands = selected,
              seeds = seeds,
              parse = parse_lt,
              meta = META,
              smoke = SMOKE)
    show_status()
    compare_dataset(dataset)

# Paper vs reproduction for every dataset/horizon (+ Avg = Table 2), from saved runs
def comparison_table():
    runs = R.load_runs(ROOT, smoke = SMOKE)
    horizon_of = {R.arg(c, "model_id"): int(R.arg(c, "pred_len")) for c in PAPER_COMMANDS}
    rows = []
    for ds, horizons in PAPER_T13.items():
        sub_ds = runs[runs.key.map(dataset_of) == ds] if len(runs) else runs
        for h, (p_mse, p_mae) in list(horizons.items()) + [("Avg", PAPER_T13_AVG[ds])]:
            if h == "Avg":
                counts = sub_ds.groupby("seed").size() if len(sub_ds) else pd.Series(dtype = int)
                complete = counts[counts == 4].index        # seeds with all 4 horizons
                per_seed = (sub_ds[sub_ds.seed.isin(complete)].groupby("seed")[["mse", "mae"]]
                            .mean() if len(complete) else None)
            else:
                sub = sub_ds[sub_ds.key.map(horizon_of) == h] if len(sub_ds) else sub_ds
                per_seed = sub.set_index("seed")[["mse", "mae"]].sort_index() if len(sub) else None
            row = {"dataset": ds, "horizon": h, "paper MSE": p_mse}
            if per_seed is not None and len(per_seed):
                for metric, paper in [("MSE", p_mse), ("MAE", p_mae)]:
                    vals = per_seed[metric.lower()]
                    if metric == "MAE":
                        row["paper MAE"] = p_mae
                    row[f"{metric} mean"] = vals.mean()
                    row[f"{metric} std"] = vals.std(ddof = 0)
                    row[f"{metric} Δ %"] = 100 * (vals.mean() - paper) / paper
                row["per seed (MSE)"] = ", ".join(f"{s}: {v:.3f}" for s, v in per_seed.mse.items())
                row["seeds"] = len(per_seed)
                row["verdict"] = R.verdict(None, max(abs(row["MSE Δ %"]), abs(row["MAE Δ %"])), "rel")
            else:
                row.update({"paper MAE": p_mae, "verdict": "not run"})
            rows.append(row)
    cols = ["paper MSE", "MSE mean", "MSE std", "MSE Δ %", "paper MAE", "MAE mean", "MAE std",
            "MAE Δ %", "per seed (MSE)", "seeds", "verdict"]
    return pd.DataFrame(rows).set_index(["dataset", "horizon"]).reindex(columns = cols)

# Shows the paper comparison for one dataset (can be called at any time)
def compare_dataset(dataset):
    table = comparison_table().loc[[dataset]]
    print(f"\n{dataset} vs paper (Table 13; Avg row = Table 2). "
          f"Verdict: within 2% = matches, within 5% = close.")
    display(table.round(3))
    return table

# Shows which (dataset, seed) pairs are finished, and how long they took
def show_status():
    runs_dir = os.path.join(ROOT, "smoke_runs" if SMOKE else "runs")
    done = [R.load_json(p) for p in glob.glob(os.path.join(runs_dir, "*.json"))]
    rows = []
    for ds in PAPER_T13:
        row = {"dataset": ds}
        for seed in SEEDS:
            mine = [d for d in done if dataset_of(d["key"]) == ds and d["seed"] == seed]
            row[f"seed {seed}"] = "✓ done" if len(mine) == 4 else f"{len(mine)}/4"
        row["GPU hours so far"] = round(sum(d["minutes"] for d in done
                                            if dataset_of(d["key"]) == ds) / 60, 2)
        rows.append(row)
    display(pd.DataFrame(rows).set_index("dataset"))

show_status()

,seed 2021,seed 2022,seed 2023,GPU hours so far
dataset,,,,
ETTm1,0/4,0/4,0/4,0.0
ETTm2,0/4,0/4,0/4,0.0
ETTh1,0/4,0/4,0/4,0.0
ETTh2,0/4,0/4,0/4,0.0
Electricity,0/4,0/4,0/4,0.0
Traffic,0/4,0/4,0/4,0.0
Weather,0/4,0/4,0/4,0.0
Exchange,0/4,0/4,0/4,0.0
ILI,0/4,0/4,0/4,0.0


#### ETTh1
Small and fast: minutes on any GPU.

In [ ]:
run_dataset("ETTh1")

== ETTh1: 4 horizon(s) x 3 seed(s) ==
[1/12] ETTh1_96_96 seed 2021: running ... done in 2.6 min -> mse 0.4048, mae 0.4204
[2/12] ETTh1_96_192 seed 2021: running ... done in 3.7 min -> mse 0.4763, mae 0.4658
[3/12] ETTh1_96_336 seed 2021: running ... done in 5.4 min -> mse 0.5436, mae 0.5060
[4/12] ETTh1_96_720 seed 2021: running ... done in 5.8 min -> mse 0.5308, mae 0.5002
[5/12] ETTh1_96_96 seed 2022: running ... done in 4.1 min -> mse 0.4053, mae 0.4190
[6/12] ETTh1_96_192 seed 2022: running ... done in 5.0 min -> mse 0.4718, mae 0.4627
[7/12] ETTh1_96_336 seed 2022: running ... done in 3.7 min -> mse 0.5072, mae 0.4798
[8/12] ETTh1_96_720 seed 2022: running ... done in 6.4 min -> mse 0.5211, mae 0.4921
[9/12] ETTh1_96_96 seed 2023: running ... done in 3.4 min -> mse 0.4051, mae 0.4210
[10/12] ETTh1_96_192 seed 2023: running ... done in 5.2 min -> mse 0.4591, mae 0.4543
[11/12] ETTh1_96_336 seed 2023: running ... done in 6.5 min -> mse 0.5065, mae 0.4827
[12/12] ETTh1_96_720 seed 20

,seed 2021,seed 2022,seed 2023,GPU hours so far
dataset,,,,
ETTm1,0/4,0/4,0/4,0.0
ETTm2,0/4,0/4,0/4,0.0
ETTh1,✓ done,✓ done,✓ done,1.0
ETTh2,0/4,0/4,0/4,0.0
Electricity,0/4,0/4,0/4,0.0
Traffic,0/4,0/4,0/4,0.0
Weather,0/4,0/4,0/4,0.0
Exchange,0/4,0/4,0/4,0.0
ILI,0/4,0/4,0/4,0.0



ETTh1 vs paper (Table 13; Avg row = Table 2). Verdict: within 2% = matches, within 5% = close.


paper MSE  MSE mean  MSE std  MSE Δ %  paper MAE  MAE mean  \
dataset horizon                                                               
ETTh1   96           0.384     0.405    0.000    5.485      0.402     0.420   
        192          0.436     0.469    0.007    7.591      0.429     0.461   
        336          0.491     0.519    0.017    5.727      0.469     0.490   
        720          0.521     0.514    0.018   -1.392      0.500     0.490   
        Avg          0.458     0.477    0.010    4.096      0.450     0.465   

                 MAE std  MAE Δ %                         per seed (MSE)  \
dataset horizon                                                            
ETTh1   96         0.001    4.512  2021: 0.405, 2022: 0.405, 2023: 0.405   
        192        0.005    7.449  2021: 0.476, 2022: 0.472, 2023: 0.459   
        336        0.012    4.379  2021: 0.544, 2022: 0.507, 2023: 0.506   
        720        0.009   -1.936  2021: 0.531, 2022: 0.521, 2023: 0.489   
        Avg        0.006    3.386  2021: 0.489, 2022: 0.476, 2023: 0.465   

                 seeds    verdict  
dataset horizon                    
ETTh1   96         3.0  ✗ differs  
        192        3.0  ✗ differs  
        336        3.0  ✗ differs  
        720        3.0  ✓ matches  
        Avg        3.0    ~ close

In [ ]:
run_dataset("ETTh1")

== ETTh1: 4 horizon(s) x 3 seed(s) ==
[1/12] ETTh1_96_96 seed 2021: running ... done in 3.6 min -> mse 0.3891, mae 0.4120
[2/12] ETTh1_96_192 seed 2021: running ... done in 3.5 min -> mse 0.4394, mae 0.4417
[3/12] ETTh1_96_336 seed 2021: running ... done in 2.9 min -> mse 0.4940, mae 0.4708
[4/12] ETTh1_96_720 seed 2021: running ... done in 4.7 min -> mse 0.5218, mae 0.4960
[5/12] ETTh1_96_96 seed 2022: running ... done in 4.0 min -> mse 0.4153, mae 0.4320
[6/12] ETTh1_96_192 seed 2022: running ... done in 4.4 min -> mse 0.4421, mae 0.4424
[7/12] ETTh1_96_336 seed 2022: running ... done in 3.9 min -> mse 0.5045, mae 0.4808
[8/12] ETTh1_96_720 seed 2022: running ... done in 5.5 min -> mse 0.5240, mae 0.4980
[9/12] ETTh1_96_96 seed 2023: running ... done in 3.6 min -> mse 0.4229, mae 0.4333
[10/12] ETTh1_96_192 seed 2023: running ... done in 4.5 min -> mse 0.4642, mae 0.4564
[11/12] ETTh1_96_336 seed 2023: running ... done in 3.4 min -> mse 0.4978, mae 0.4725
[12/12] ETTh1_96_720 seed 20

,seed 2021,seed 2022,seed 2023,GPU hours so far
dataset,,,,
ETTm1,0/4,0/4,0/4,0.0
ETTm2,0/4,0/4,0/4,0.0
ETTh1,✓ done,✓ done,✓ done,0.8
ETTh2,0/4,0/4,0/4,0.0
Electricity,0/4,0/4,0/4,0.0
Traffic,0/4,0/4,0/4,0.0
Weather,0/4,0/4,0/4,0.0
Exchange,0/4,0/4,0/4,0.0
ILI,0/4,0/4,0/4,0.0



ETTh1 vs paper (Table 13; Avg row = Table 2). Verdict: within 2% = matches, within 5% = close.


paper MSE  MSE mean  MSE std  MSE Δ %  paper MAE  MAE mean  \
dataset horizon                                                               
ETTh1   96           0.384     0.409    0.014    6.532      0.402     0.426   
        192          0.436     0.449    0.011    2.880      0.429     0.447   
        336          0.491     0.499    0.004    1.579      0.469     0.475   
        720          0.521     0.524    0.002    0.606      0.500     0.498   
        Avg          0.458     0.470    0.007    2.650      0.450     0.461   

                 MAE std  MAE Δ %                         per seed (MSE)  \
dataset horizon                                                            
ETTh1   96         0.010    5.911  2021: 0.389, 2022: 0.415, 2023: 0.423   
        192        0.007    4.155  2021: 0.439, 2022: 0.442, 2023: 0.464   
        336        0.004    1.213  2021: 0.494, 2022: 0.505, 2023: 0.498   
        720        0.002   -0.329  2021: 0.522, 2022: 0.524, 2023: 0.527   
        Avg        0.005    2.535  2021: 0.461, 2022: 0.471, 2023: 0.478   

                 seeds    verdict  
dataset horizon                    
ETTh1   96         3.0  ✗ differs  
        192        3.0    ~ close  
        336        3.0  ✓ matches  
        720        3.0  ✓ matches  
        Avg        3.0    ~ close

#### ETTh2
Small and fast.

In [ ]:
run_dataset("ETTh2")

== ETTh2: 4 horizon(s) x 3 seed(s) ==
[1/12] ETTh2_96_96 seed 2021: running ... done in 2.3 min -> mse 0.3370, mae 0.3709
[2/12] ETTh2_96_192 seed 2021: running ... done in 7.7 min -> mse 0.4046, mae 0.4146
[3/12] ETTh2_96_336 seed 2021: running ... done in 11.0 min -> mse 0.4538, mae 0.4514
[4/12] ETTh2_96_720 seed 2021: running ... done in 9.8 min -> mse 0.4335, mae 0.4477
[5/12] ETTh2_96_96 seed 2022: running ... done in 4.0 min -> mse 0.3191, mae 0.3613
[6/12] ETTh2_96_192 seed 2022: running ... done in 5.5 min -> mse 0.4130, mae 0.4156
[7/12] ETTh2_96_336 seed 2022: running ... done in 3.5 min -> mse 0.4554, mae 0.4510
[8/12] ETTh2_96_720 seed 2022: running ... done in 14.8 min -> mse 0.4722, mae 0.4719
[9/12] ETTh2_96_96 seed 2023: running ... done in 4.1 min -> mse 0.3200, mae 0.3639
[10/12] ETTh2_96_192 seed 2023: running ... done in 5.6 min -> mse 0.3971, mae 0.4068
[11/12] ETTh2_96_336 seed 2023: running ... done in 7.4 min -> mse 0.4494, mae 0.4510
[12/12] ETTh2_96_720 seed 

,seed 2021,seed 2022,seed 2023,GPU hours so far
dataset,,,,
ETTm1,0/4,0/4,0/4,0.00
ETTm2,0/4,0/4,0/4,0.00
ETTh1,✓ done,✓ done,✓ done,1.00
ETTh2,✓ done,✓ done,✓ done,1.41
Electricity,0/4,0/4,0/4,0.00
Traffic,0/4,0/4,0/4,0.00
Weather,0/4,0/4,0/4,0.00
Exchange,✓ done,✓ done,✓ done,0.80
ILI,0/4,0/4,0/4,0.00



ETTh2 vs paper (Table 13; Avg row = Table 2). Verdict: within 2% = matches, within 5% = close.


paper MSE  MSE mean  MSE std  MSE Δ %  paper MAE  MAE mean  \
dataset horizon                                                               
ETTh2   96           0.340     0.325    0.008   -4.301      0.374     0.365   
        192          0.402     0.405    0.007    0.729      0.414     0.412   
        336          0.452     0.453    0.003    0.188      0.452     0.451   
        720          0.462     0.458    0.018   -0.810      0.468     0.464   
        Avg          0.414     0.410    0.003   -0.881      0.427     0.423   

                 MAE std  MAE Δ %                         per seed (MSE)  \
dataset horizon                                                            
ETTh2   96         0.004   -2.303  2021: 0.337, 2022: 0.319, 2023: 0.320   
        192        0.004   -0.398  2021: 0.405, 2022: 0.413, 2023: 0.397   
        336        0.000   -0.192  2021: 0.454, 2022: 0.455, 2023: 0.449   
        720        0.011   -0.908  2021: 0.434, 2022: 0.472, 2023: 0.469   
        Avg        0.002   -0.900  2021: 0.407, 2022: 0.415, 2023: 0.409   

                 seeds    verdict  
dataset horizon                    
ETTh2   96         3.0    ~ close  
        192        3.0  ✓ matches  
        336        3.0  ✓ matches  
        720        3.0  ✓ matches  
        Avg        3.0  ✓ matches

#### ETTm1
Small, 4x more data than ETTh1: well under an hour.

In [ ]:
run_dataset("ETTm1")

== ETTm1: 4 horizon(s) x 3 seed(s) ==
[1/12] ETTm1_96_96 seed 2021: running ... done in 17.0 min -> mse 0.3339, mae 0.3754
[2/12] ETTm1_96_192 seed 2021: running ... done in 26.8 min -> mse 0.4078, mae 0.4136
[3/12] ETTm1_96_336 seed 2021: running ...

KeyboardInterrupt: 

#### ETTm2
Small, 4x more data than ETTh1: well under an hour.

In [ ]:
run_dataset("ETTm2")

#### Exchange
Tiny (1 training epoch): minutes.

In [ ]:
run_dataset("Exchange")

== Exchange: 4 horizon(s) x 3 seed(s) ==
[1/12] Exchange_96_96 seed 2021: already done, skipping
[2/12] Exchange_96_192 seed 2021: already done, skipping
[3/12] Exchange_96_336 seed 2021: already done, skipping
[4/12] Exchange_96_720 seed 2021: running ... done in 8.3 min -> mse 0.9294, mae 0.7334
[5/12] Exchange_96_96 seed 2022: running ... done in 1.8 min -> mse 0.1071, mae 0.2346
[6/12] Exchange_96_192 seed 2022: running ... done in 4.0 min -> mse 0.2103, mae 0.3306
[7/12] Exchange_96_336 seed 2022: running ... done in 3.1 min -> mse 0.3666, mae 0.4399
[8/12] Exchange_96_720 seed 2022: running ... done in 3.1 min -> mse 0.9170, mae 0.7331
[9/12] Exchange_96_96 seed 2023: running ... done in 3.3 min -> mse 0.1062, mae 0.2354
[10/12] Exchange_96_192 seed 2023: running ... done in 3.7 min -> mse 0.2044, mae 0.3266
[11/12] Exchange_96_336 seed 2023: running ... done in 4.2 min -> mse 0.3673, mae 0.4422
[12/12] Exchange_96_720 seed 2023: running ... done in 4.0 min -> mse 0.9356, mae 0.7

,seed 2021,seed 2022,seed 2023,GPU hours so far
dataset,,,,
ETTm1,0/4,0/4,0/4,0.0
ETTm2,0/4,0/4,0/4,0.0
ETTh1,✓ done,✓ done,✓ done,1.0
ETTh2,0/4,0/4,0/4,0.0
Electricity,0/4,0/4,0/4,0.0
Traffic,0/4,0/4,0/4,0.0
Weather,0/4,0/4,0/4,0.0
Exchange,✓ done,✓ done,✓ done,0.8
ILI,0/4,0/4,0/4,0.0



Exchange vs paper (Table 13; Avg row = Table 2). Verdict: within 2% = matches, within 5% = close.


paper MSE  MSE mean  MSE std  MSE Δ %  paper MAE  MAE mean  \
dataset  horizon                                                               
Exchange 96           0.107     0.109    0.003    1.903      0.234     0.237   
         192          0.226     0.208    0.003   -7.944      0.344     0.329   
         336          0.367     0.367    0.000    0.041      0.448     0.443   
         720          0.964     0.927    0.008   -3.803      0.746     0.734   
         Avg          0.416     0.403    0.002   -3.151      0.443     0.436   

                  MAE std  MAE Δ %                         per seed (MSE)  \
dataset  horizon                                                            
Exchange 96         0.003    1.257  2021: 0.114, 2022: 0.107, 2023: 0.106   
         192        0.002   -4.333  2021: 0.209, 2022: 0.210, 2023: 0.204   
         336        0.003   -1.155  2021: 0.368, 2022: 0.367, 2023: 0.367   
         720        0.001   -1.646  2021: 0.929, 2022: 0.917, 2023: 0.936   
         Avg        0.001   -1.660  2021: 0.405, 2022: 0.400, 2023: 0.403   

                  seeds    verdict  
dataset  horizon                    
Exchange 96         3.0  ✓ matches  
         192        3.0  ✗ differs  
         336        3.0  ✓ matches  
         720        3.0    ~ close  
         Avg        3.0    ~ close

#### ILI
Tiny dataset: minutes.

In [ ]:
run_dataset("ILI")

== ILI: 4 horizon(s) x 3 seed(s) ==
[1/12] ili_36_24 seed 2021: running ...


KeyboardInterrupt



#### Weather
21 variables, 1 training epoch: well under an hour.

In [ ]:
run_dataset("Weather")

#### Electricity
**Heavy**: roughly 4-7 h per seed on G4, 15-25 h on L4 (estimates). Consider one seed per day, e.g. `run_dataset("Electricity", seeds=[2021])`.

In [ ]:
run_dataset("Electricity")

#### Traffic
**Heaviest**: roughly 5-10 h per seed on G4, 20-30 h on L4 (estimates). Split it further if needed, e.g. `run_dataset("Traffic", seeds=[2021], horizons=[96, 192])` one day and `horizons=[336, 720]` the next (720 takes about 4x longer than 96).

In [ ]:
run_dataset("Traffic")

## 7. Comparison with the paper

In [ ]:
table = comparison_table()
assert (table.verdict != "not run").any(), "No finished runs yet - run section 6 first."
table.round(4).to_csv(os.path.join(ROOT, "table13_reproduction.csv"))
open(os.path.join(ROOT, "table13_reproduction.md"), "w").write(table.round(4).to_markdown())
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 20)
table.round(3)

## 8. Determinism check
Re-runs one configuration and confirms the result is identical to the stored one.

In [ ]:
# Re-runs ETTh1 horizon 96 with seed 2021 and checks MSE/MAE are identical.
etth1 = [c for c in COMMANDS if R.arg(c, "model_id") == "ETTh1_96_96"][0]
R.determinism_check(PY, REPO, ROOT, etth1, SEEDS[0], parse_lt, smoke=SMOKE)